
# Log-likelihood scoring vs. `generate_until`
### How MMLU / HellaSwag / ARC actually work — and why they fit on an 8GB laptop while GSM8K-style generation doesn't

This notebook is a hands-on companion to the accompanying `toy_lm.py` and
`lesson.html`. No GPU, no `transformers`, no downloads — everything here
runs on a **hand-built toy language model** (a bigram table you can print
and read) so every number is fully visible.

The scoring *math* you'll see here is **identical** to what a real eval
harness (e.g. EleutherAI's `lm-evaluation-harness`) does against a real
7B-parameter model — we've just shrunk the box that produces
`P(next_token | context)` down to something you can hold in your head.

**Run the cells top to bottom.** Change numbers, break things, see what happens.



## 1. The one idea everything else builds on

A language model, at its core, is a function:

$$
f(\text{context}) \;\longrightarrow\; \text{logits} \in \mathbb{R}^{|V|}
$$

Feed it a sequence of tokens, it hands back one score per vocabulary word —
"how much do I want to say this word next." Turn those into probabilities
with softmax:

$$
P(v \mid \text{context}) = \frac{e^{\,\text{logit}_v}}{\sum_{v'} e^{\,\text{logit}_{v'}}}
$$

**Everything below — MMLU, HellaSwag, ARC, GSM8K — is just different ways of
calling this one function repeatedly.**


In [1]:

import sys
sys.path.insert(0, '.')
from toy_lm import *
import numpy as np

model = ToyLM.hand_built()
print(f"Vocab size: {V}")
print(f"Vocab: {VOCAB}")


Vocab size: 24
Vocab: ['<bos>', 'the', 'capital', 'of', 'france', 'germany', 'italy', 'is', 'paris', 'berlin', 'rome', 'madrid', 'a', 'city', 'in', 'europe', 'answer', ':', 'yes', 'no', '42', '5', '7', '12']


In [2]:

# The model's opinion about what comes after "is", given a tiny bigram model
context_ids = tokenize("<bos> the capital of france is")
logprobs = model.next_token_logprobs(context_ids)

# show the top few candidates
order = np.argsort(-logprobs)[:8]
for i in order:
    print(f"  P({ID2TOK[i]!r:>10} | ...is) = {np.exp(logprobs[i]):.4f}   (logP = {logprobs[i]:.4f})")


  P(   'paris' | ...is) = 0.7910   (logP = -0.2344)
  P(  'berlin' | ...is) = 0.1071   (logP = -2.2344)
  P(    'rome' | ...is) = 0.0649   (logP = -2.7344)
  P(       'a' | ...is) = 0.0145   (logP = -4.2344)
  P(  'madrid' | ...is) = 0.0088   (logP = -4.7344)
  P(   '<bos>' | ...is) = 0.0007   (logP = -7.2344)
  P( 'capital' | ...is) = 0.0007   (logP = -7.2344)
  P(     'the' | ...is) = 0.0007   (logP = -7.2344)



Notice this is a **single forward pass** — one call to the model — that
returns a probability for *every* word in the vocabulary at once. The model
never "chose" anything here; we just read off its opinion.



## 2. `loglikelihood`: scoring a *given* string (MMLU / HellaSwag / ARC)

Multiple-choice benchmarks never ask the model to write an answer. They
give it a **fixed set of candidate strings** and ask: *"which one would you
have been least surprised to produce?"*

For a continuation $c = c_1, c_2, \dots, c_n$ (tokenized) following some
context, the score is:

$$
\text{score}(c) = \sum_{i=1}^{n} \log P(c_i \mid \text{context}, c_1, \dots, c_{i-1})
$$

Sum of log-probabilities, not product of probabilities — logs turn a chain
of small multiplications into an addition, which is both numerically
stable (no underflow from multiplying hundreds of numbers < 1) and easy to
reason about term by term.

No sampling. No generation loop. No stopping condition. Just: tokenize the
candidate, walk down it, and add up "how expected was each token."


In [3]:

# A worked MMLU-style 4-choice question:
#   "The capital of France is ___"
#   A) paris   B) berlin   C) rome   D) madrid

context = "<bos> the capital of france is"
choices = ["paris", "berlin", "rome", "madrid"]

print(f'Context: {context!r}\n')
for choice in choices:
    score = loglikelihood(model, context, choice, verbose=True)
    print(f"  --> total log-likelihood({choice!r}) = {score:.4f}\n")


Context: '<bos> the capital of france is'

    log P('paris' | ...) = -0.2344   (P = 0.7910)
  --> total log-likelihood('paris') = -0.2344

    log P('berlin' | ...) = -2.2344   (P = 0.1071)
  --> total log-likelihood('berlin') = -2.2344

    log P('rome' | ...) = -2.7344   (P = 0.0649)
  --> total log-likelihood('rome') = -2.7344

    log P('madrid' | ...) = -4.7344   (P = 0.0088)
  --> total log-likelihood('madrid') = -4.7344



In [4]:

best_idx, scores = multiple_choice_score(model, context, choices)
labels = ["A", "B", "C", "D"]
print("Final scores:")
for lab, ch, s in zip(labels, choices, scores):
    marker = "  <-- picked" if ch == choices[best_idx] else ""
    print(f"  {lab}) {ch:8s}  score = {s:8.4f}{marker}")


Final scores:
  A) paris     score =  -0.2344  <-- picked
  B) berlin    score =  -2.2344
  C) rome      score =  -2.7344
  D) madrid    score =  -4.7344



### Why this is one forward pass per choice, not per token

For a single-token answer like `"paris"`, the whole loop above is just:

1. Run the model once on `context`.
2. Read off `logP("paris" | context)` from that single output vector.

For a **multi-token** answer, you can't shortcut it — each token's
probability depends on all the tokens before it (including earlier tokens
*of the same continuation*), so you must feed them in one at a time. Let's
see that with a longer continuation:


In [5]:

# Multi-token continuation: "answer : 7" vs "answer : 42"
print("Scoring 'answer : 7':")
s7 = loglikelihood(model, "<bos>", "answer : 7", verbose=True)
print(f"  total = {s7:.4f}\n")

print("Scoring 'answer : 42':")
s42 = loglikelihood(model, "<bos>", "answer : 42", verbose=True)
print(f"  total = {s42:.4f}\n")

print(f"'7' wins by {s7 - s42:.4f} log-prob units "
      f"({np.exp(s7 - s42):.2f}x more likely as a continuation)")


Scoring 'answer : 7':
    log P('answer' | ...) = -5.1441   (P = 0.0058)
    log P(':' | ...) = -0.0208   (P = 0.9795)
    log P('7' | ...) = -0.8397   (P = 0.4319)
  total = -6.0045

Scoring 'answer : 42':
    log P('answer' | ...) = -5.1441   (P = 0.0058)
    log P(':' | ...) = -0.0208   (P = 0.9795)
    log P('42' | ...) = -2.3397   (P = 0.0964)
  total = -7.5045

'7' wins by 1.5000 log-prob units (4.48x more likely as a continuation)



Look closely at the trace above: `log P('answer' | ...)` and
`log P(':' | ...)` are **identical in both runs** — same context up to that
point, so same probability. Only the last token differs. This is the
detail that makes loglikelihood scoring cheap: for $n$ answer choices that
share a long common prefix (the question), a smart implementation caches
the shared prefix computation and only re-does the diverging tail. (The
toy model above recomputes everything for clarity; real harnesses exploit
this for speed.)



## 3. `generate_until`: producing a string you don't know in advance (GSM8K)

GSM8K asks a math word problem and wants a free-form chain-of-reasoning
answer: *"Natalia sold clips to 48 friends in April, then half as many in
May. How many total?"* There is no fixed list of candidate strings to score
— the model must **generate** its answer, token by token:

```
loop until stop condition or max_tokens:
    1. forward pass on (context + tokens generated so far)
    2. pick next token (argmax for greedy, or sample)
    3. append it to the sequence
    4. check: does the text so far contain a stop string? if so, stop.
```

Critically: **each iteration of this loop is its own forward pass**, and
each pass's *input* is one token longer than the last one. For a 50-token
answer, that's 50 forward passes (or 50 incremental ones, reusing the KV
cache — more on that in Part 4).


In [6]:

result = generate_until(
    model,
    context="<bos> the capital of france is",
    stop_strings=["paris", "berlin", "rome"],
    max_new_tokens=10,
    verbose=True,
)
print(f"\nFinal generated text: {result!r}")


    step 1: forward pass -> pick 'paris' (logP=-0.234)  | text so far: '<bos> the capital of france is paris'

Final generated text: '<bos> the capital of france is paris'


In [7]:

# A longer generation to make the step-by-step cost visible
result = generate_until(
    model,
    context="<bos> the capital of germany is berlin a city in europe answer :",
    stop_strings=["7", "42", "5", "12"],
    max_new_tokens=15,
    verbose=True,
)
print(f"\nFinal: {result!r}")


    step 1: forward pass -> pick '7' (logP=-0.840)  | text so far: '<bos> the capital of germany is berlin a city in europe answer : 7'

Final: '<bos> the capital of germany is berlin a city in europe answer : 7'



Compare the two verbose traces you've now seen:

| | `loglikelihood` | `generate_until` |
|---|---|---|
| What's known in advance | The exact candidate string | Nothing — output is produced live |
| # forward passes | 1 per candidate (batched: 1 pass covers all tokens of that candidate, in real models) | 1 **per output token**, sequential, can't be parallelized because token $t{+}1$ needs token $t$ |
| Output length | Fixed (you already have the string) | Unknown — must generate until a stop condition fires or a cap is hit |
| Used by | MMLU, HellaSwag, ARC, (any multiple-choice / cloze task) | GSM8K, any free-form-answer task |



## 4. Why one fits an 8GB laptop and the other OOMs: the KV cache

Here's the mechanical reason generation is memory-hungrier than scoring,
even though both are "just forward passes."

A transformer's self-attention needs, at every layer, the **Key** and
**Value** vectors for *every token seen so far* — that's the whole point of
attention (each new token attends back over all previous ones). Recomputing
K/V for the entire history at every single generation step would be
wasteful, so real inference engines cache them: the **KV cache**.

$$
\text{KV cache bytes} = 2 \times L \times H \times d_{head} \times S \times B \times \text{dtype\_bytes}
$$

- $L$ = number of transformer layers
- $H$ = number of attention heads
- $d_{head}$ = dimension per head
- $S$ = sequence length **so far**
- $B$ = batch size
- the leading $2$ is for **K** *and* **V**
- `dtype_bytes` = 2 for fp16/bf16, 4 for fp32

The crucial variable is $S$: **the cache grows linearly with every new
token you generate**, and it grows on top of the (fixed) model weights.


In [8]:

# Rough config for a Llama-2-7B-class model
n_layers, n_heads, head_dim = 32, 32, 128
model_weights_gb = 7 * 2 / 1024   # ~7B params * 2 bytes (fp16) -> GB (rough)
print(f"Model weights alone (fp16): ~{model_weights_gb:.2f} GB\n")

for seq_len in [4, 32, 128, 512, 2048, 8192]:
    kv_bytes = kv_cache_bytes(n_layers, n_heads, head_dim, seq_len)
    total_gb = model_weights_gb + kv_bytes / 1024**3
    print(f"seq_len={seq_len:5d} tokens  ->  KV cache = {human_bytes(kv_bytes):>10}  "
          f"|  weights+KV = {total_gb:6.2f} GB")


Model weights alone (fp16): ~0.01 GB

seq_len=    4 tokens  ->  KV cache =    2.00 MB  |  weights+KV =   0.02 GB
seq_len=   32 tokens  ->  KV cache =   16.00 MB  |  weights+KV =   0.03 GB
seq_len=  128 tokens  ->  KV cache =   64.00 MB  |  weights+KV =   0.08 GB
seq_len=  512 tokens  ->  KV cache =  256.00 MB  |  weights+KV =   0.26 GB
seq_len= 2048 tokens  ->  KV cache =    1.00 GB  |  weights+KV =   1.01 GB
seq_len= 8192 tokens  ->  KV cache =    4.00 GB  |  weights+KV =   4.01 GB



### Reading this table against an 8GB laptop

- **`loglikelihood` scoring** (MMLU/HellaSwag/ARC): the context is a
  question plus one short answer choice — typically well under a few
  hundred tokens. Looking at the table, that's tens to low-hundreds of MB
  of KV cache on top of the model weights. On an 8GB machine (running a
  quantized model, e.g. 4-bit, to fit weights in ~4GB), you have plenty of
  headroom left.

- **`generate_until`** (GSM8K-style chain-of-thought): the model may need
  to generate **hundreds of tokens of reasoning** before it reaches an
  answer, and — this is the part people trip on — if you're evaluating a
  **batch** of examples at once for throughput, the KV cache cost
  multiplies by batch size $B$ too. Look at `seq_len=2048`: nearly 1 GB of
  cache *per sequence in the batch*, on top of the weights. Try to batch 8
  long GSM8K generations together and you can blow past 8GB fast — even
  though the model weights themselves haven't changed size at all.

**The core asymmetry:** loglikelihood scoring has a *bounded*, *known in
advance* sequence length (you already have the candidate string). Free-form
generation has an *open-ended* sequence length that you discover by running
the model — so the worst case (a model that rambles for 1000+ tokens before
emitting the stop string) is exactly the case that exhausts memory on
constrained hardware.


In [9]:

# Try it yourself: at what sequence length does a batch of 8 blow past 8GB
# on an unquantized-weights budget?
budget_gb = 8.0
for batch in [1, 4, 8, 16]:
    print(f"batch_size={batch}")
    for seq_len in [128, 512, 1024, 2048, 4096]:
        kv_gb = kv_cache_bytes(n_layers, n_heads, head_dim, seq_len, batch_size=batch) / 1024**3
        flag = "  <-- exceeds 8GB budget alone!" if kv_gb > budget_gb else ""
        print(f"    seq_len={seq_len:5d}  KV cache = {kv_gb:6.3f} GB{flag}")


batch_size=1
    seq_len=  128  KV cache =  0.062 GB
    seq_len=  512  KV cache =  0.250 GB
    seq_len= 1024  KV cache =  0.500 GB
    seq_len= 2048  KV cache =  1.000 GB
    seq_len= 4096  KV cache =  2.000 GB
batch_size=4
    seq_len=  128  KV cache =  0.250 GB
    seq_len=  512  KV cache =  1.000 GB
    seq_len= 1024  KV cache =  2.000 GB
    seq_len= 2048  KV cache =  4.000 GB
    seq_len= 4096  KV cache =  8.000 GB
batch_size=8
    seq_len=  128  KV cache =  0.500 GB
    seq_len=  512  KV cache =  2.000 GB
    seq_len= 1024  KV cache =  4.000 GB
    seq_len= 2048  KV cache =  8.000 GB
    seq_len= 4096  KV cache = 16.000 GB  <-- exceeds 8GB budget alone!
batch_size=16
    seq_len=  128  KV cache =  1.000 GB
    seq_len=  512  KV cache =  4.000 GB
    seq_len= 1024  KV cache =  8.000 GB
    seq_len= 2048  KV cache = 16.000 GB  <-- exceeds 8GB budget alone!
    seq_len= 4096  KV cache = 32.000 GB  <-- exceeds 8GB budget alone!



## 5. Play with it

A few things to try, using only what's defined above:

1. **Change the model's confidence.** Open `toy_lm.py`, find `hand_built()`,
   and change `boost("is", "paris", 5.0)` to a smaller number. Re-run the
   MMLU cell — does the model still pick the right answer? At what point
   does it flip to a wrong one?

2. **Build your own 4-choice question.** Using only words in `VOCAB`
   (print it if you forget), write a new `context` + `choices` and call
   `multiple_choice_score`.

3. **Force a long generation.** Call `generate_until` with a `stop_strings`
   list that never matches anything in the vocab, and watch `max_new_tokens`
   become the only thing that stops it — this is precisely the runaway
   case that costs memory in real generation.

4. **Compute your own memory budget.** Using `kv_cache_bytes`, work out how
   long a sequence a 13B model (`n_layers=40, n_heads=40, head_dim=128`)
   could generate on your own machine's free RAM before the KV cache alone
   exceeds it.


In [10]:

# Scratch space — try the exercises above here.




## 6. One-paragraph summary

Multiple-choice benchmarks (MMLU, HellaSwag, ARC) never generate text: they
score every candidate answer by summing `log P(token | context)` across the
tokens of that candidate, then pick the highest-scoring one — a small,
fixed number of forward passes over short, known-length sequences.
Free-form benchmarks (GSM8K) generate text one token at a time, feeding
each new token back into the model, with no upper bound on length known in
advance except an artificial cap — and each generated token grows a KV
cache that scales with sequence length (and batch size). That's the whole
reason the first fits comfortably on modest hardware and the second can
quietly eat all of it.
